# Echo-Check — LOF Encoder GPU Benchmark

Tests the LOF encoder ONNX models (FP32 and INT8) on both CPU and GPU.

**Models under test:**
- `encoder_simplified.onnx` — FP32, graph-optimised
- `encoder_int8.onnx` — INT8 static quantised (QDQ, 74.4% smaller)

**Architecture:** Conv2D encoder -> 128-dim embedding  
**Input:** `(1, 1, 128, 432)` mel spectrogram  
**Output:** `(1, 128)` latent embedding -> scored by LOF


## Step 1: Installing dependencies

In [1]:
!pip install -q onnx onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.6 MB/s eta 0:00:00


## Step 2: Uploading ONNX files

Upload these two files from `phase3_outputs_lof/`:
- `encoder_simplified.onnx`
- `encoder_int8.onnx`

In [2]:
from google.colab import files

print("Upload encoder_simplified.onnx and encoder_int8.onnx")
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

Upload encoder_simplified.onnx and encoder_int8.onnx


Saving encoder_int8.onnx to encoder_int8.onnx
Saving encoder_simplified.onnx to encoder_simplified.onnx

Uploaded: ['encoder_int8.onnx', 'encoder_simplified.onnx']


## Step 3: Checking GPU

In [3]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("No GPU detected — make sure runtime is set to GPU T4")

Thu Apr 16 00:25:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 4: Loading models and inspect

In [9]:
import onnx
import onnxruntime as ort
import numpy as np
import time

FP32_PATH = "encoder_simplified.onnx"
INT8_PATH = "encoder_int8.onnx"

#Reading input shape directly from ONNX model
model_fp32 = onnx.load(FP32_PATH)
input_shape = [d.dim_value if d.dim_value > 0 else 1
               for d in model_fp32.graph.input[0].type.tensor_type.shape.dim]
input_shape[0] = 1  # batch size = 1
print(f"Input shape  : {input_shape}")

output_shape = [d.dim_value if d.dim_value > 0 else 1
                for d in model_fp32.graph.output[0].type.tensor_type.shape.dim]
output_shape[0] = 1
print(f"Output shape : {output_shape}  (embedding dim = {output_shape[-1]})")

import os
fp32_kb = os.path.getsize(FP32_PATH) / 1024
int8_kb  = os.path.getsize(INT8_PATH) / 1024
print(f"\nFP32 size : {fp32_kb:.1f} KB")
print(f"INT8 size : {int8_kb:.1f} KB  ({(1 - int8_kb/fp32_kb)*100:.1f}% smaller)")

#Dummy input
dummy = np.random.rand(*input_shape).astype(np.float32)

Input shape  : [1, 1, 128, 432]
Output shape : [1, 128]  (embedding dim = 128)

FP32 size : 1646.4 KB
INT8 size : 424.6 KB  (74.2% smaller)


## Step 5: Benchmarks

In [10]:
def make_session(path, provider):
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=opts, providers=[provider])


def benchmark(session, dummy_input, n_warmup=50, n_runs=500):
    iname = session.get_inputs()[0].name

    for _ in range(n_warmup):
        session.run(None, {iname: dummy_input})

    #Timed runs
    t0 = time.perf_counter()
    for _ in range(n_runs):
        session.run(None, {iname: dummy_input})
    elapsed_ms = (time.perf_counter() - t0) / n_runs * 1000

    return elapsed_ms


print("Benchmark function ready.")
print(f"Available providers : {ort.get_available_providers()}")

Benchmark function ready.
Available providers : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## Step 6: Running all benchmarks

In [11]:
#CPU benchmarks
print("Running CPU benchmarks (500 runs each)...")

sess_cpu_fp32 = make_session(FP32_PATH, "CPUExecutionProvider")
sess_cpu_int8 = make_session(INT8_PATH, "CPUExecutionProvider")

cpu_fp32_ms = benchmark(sess_cpu_fp32, dummy)
print(f"  CPU FP32 : {cpu_fp32_ms:.3f} ms")

cpu_int8_ms = benchmark(sess_cpu_int8, dummy)
print(f"  CPU INT8 : {cpu_int8_ms:.3f} ms")

cpu_speedup = cpu_fp32_ms / cpu_int8_ms
print(f"  CPU INT8 speedup : {cpu_speedup:.2f}x")

Running CPU benchmarks (500 runs each)...
  CPU FP32 : 4.478 ms
  CPU INT8 : 4.232 ms
  CPU INT8 speedup : 1.06x


In [12]:
#GPU benchmarks
if "CUDAExecutionProvider" not in ort.get_available_providers():
    print("CUDA not available — skipping GPU benchmark.")
    gpu_fp32_ms = None
    gpu_int8_ms = None
else:
    print("Running GPU benchmarks (500 runs each)...")

    sess_gpu_fp32 = make_session(FP32_PATH, "CUDAExecutionProvider")
    sess_gpu_int8 = make_session(INT8_PATH, "CUDAExecutionProvider")

    gpu_fp32_ms = benchmark(sess_gpu_fp32, dummy)
    print(f"  GPU FP32 : {gpu_fp32_ms:.3f} ms")

    gpu_int8_ms = benchmark(sess_gpu_int8, dummy)
    print(f"  GPU INT8 : {gpu_int8_ms:.3f} ms")

    gpu_speedup = gpu_fp32_ms / gpu_int8_ms
    print(f"  GPU INT8 speedup : {gpu_speedup:.2f}x")

Running GPU benchmarks (500 runs each)...
  GPU FP32 : 0.544 ms
  GPU INT8 : 0.525 ms
  GPU INT8 speedup : 1.04x


## Step 7: Summary

In [13]:
print("=" * 62)
print("  Echo-Check LOF Encoder — CPU vs GPU Benchmark")
print("=" * 62)
print(f"  Model       : Conv2D encoder  ({output_shape[-1]}-dim embedding)")
print(f"  Input shape : {input_shape}")
print(f"  FP32 size   : {fp32_kb:.1f} KB")
print(f"  INT8 size   : {int8_kb:.1f} KB  ({(1-int8_kb/fp32_kb)*100:.1f}% smaller)")
print()
print(f"  {'Config':<25} {'Latency':>10}  {'vs CPU FP32':>12}")
print(f"  {'-'*50}")
print(f"  {'CPU  FP32':<25} {cpu_fp32_ms:>9.3f}ms  {'(baseline)':>12}")
print(f"  {'CPU  INT8':<25} {cpu_int8_ms:>9.3f}ms  {f'{cpu_fp32_ms/cpu_int8_ms:.2f}x':>12}")

if gpu_fp32_ms is not None:
    print(f"  {'GPU  FP32':<25} {gpu_fp32_ms:>9.3f}ms  {f'{cpu_fp32_ms/gpu_fp32_ms:.2f}x':>12}")
    print(f"  {'GPU  INT8':<25} {gpu_int8_ms:>9.3f}ms  {f'{cpu_fp32_ms/gpu_int8_ms:.2f}x':>12}")
else:
    print(f"  {'GPU  FP32':<25} {'N/A':>10}")
    print(f"  {'GPU  INT8':<25} {'N/A':>10}")

print(f"  {'-'*50}")
print()
print("=" * 62)

  Echo-Check LOF Encoder — CPU vs GPU Benchmark
  Model       : Conv2D encoder  (128-dim embedding)
  Input shape : [1, 1, 128, 432]
  FP32 size   : 1646.4 KB
  INT8 size   : 424.6 KB  (74.2% smaller)

  Config                       Latency   vs CPU FP32
  --------------------------------------------------
  CPU  FP32                     4.478ms    (baseline)
  CPU  INT8                     4.232ms         1.06x
  GPU  FP32                     0.544ms         8.23x
  GPU  INT8                     0.525ms         8.53x
  --------------------------------------------------

